In [4]:
import recordlinkage
import pandas as pd
import numpy as np


industry_dataset = pd.read_excel('../../Mediated Schema Excels/industry_schema.xlsx')

industry_dataset = industry_dataset[
    ~(
        industry_dataset[['IndustryName', 'Sector']].isnull().all(axis=1) | 
        industry_dataset[['IndustryName', 'Sector']].apply(lambda x: x.str.strip()).eq('').all(axis=1)
    )
]


def industry_blocking():
    indexer = recordlinkage.Index()
    indexer.block(['IndustryName', 'Sector'])
    candidate_links = indexer.index(industry_dataset)
    
    #eliminates duplicates like x and y, y and x. With this form, only x and y is stored
    candidate_links = candidate_links[~candidate_links.duplicated()]

    compare = recordlinkage.Compare()
    compare.string('IndustryName', 'IndustryName', method='jarowinkler', label='industry_similarity')
    compare.string('Sector', 'Sector', method='jarowinkler', label='sector_similarity')
    compare_vectors = compare.compute(candidate_links, industry_dataset)

    
    matched_pairs = compare_vectors[
        (compare_vectors['industry_similarity'] > 0.92) | (compare_vectors['sector_similarity'] > 0.96)
    ]


    n = len(matched_pairs)

    
    df = pd.DataFrame({
        "id": matched_pairs.index.get_level_values(0),
        "left_row_index": matched_pairs.index.get_level_values(0),
        "right_row_index": matched_pairs.index.get_level_values(1),
        "left_name_company": industry_dataset.loc[matched_pairs.index.get_level_values(0), "Name"].values,
        "right_name_company": industry_dataset.loc[matched_pairs.index.get_level_values(1), "Name"].values,
        "left_industryname": industry_dataset.loc[matched_pairs.index.get_level_values(0), "IndustryName"].values,
        "right_industryname": industry_dataset.loc[matched_pairs.index.get_level_values(1), "IndustryName"].values,
        "left_sector": industry_dataset.loc[matched_pairs.index.get_level_values(0), "Sector"].values,
        "right_sector": industry_dataset.loc[matched_pairs.index.get_level_values(1), "Sector"].values,
        "left_address": [np.nan] * n,
        "right_address": [np.nan] * n,
        "left_city": [np.nan] * n,
        "right_city": [np.nan] * n,
        "left_state": [np.nan] * n,
        "right_state": [np.nan] * n,
        "left_country": [np.nan] * n,
        "right_country": [np.nan] * n,
        "left_continent": [np.nan] * n,
        "right_continent": [np.nan] * n,
        "left_other": [np.nan] * n,
        "right_other": [np.nan] * n,
    })

    
    df.to_csv("../../Deep Matcher/Blocking and Pairwise matching/industry_blocking_for_deepmatcher.csv", index=False)


In [5]:
industry_blocking()